In [ ]:
import numpy as np
import librosa
import os, glob, joblib
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score
import warnings

In [ ]:
CANONICAL_SR = 22050
MIN_DURATION_SEC = 0.5  #rejection buffer after trim
N_MFCC = 20

In [ ]:
def extractChordFeatures(y, sr, target_sr=CANONICAL_SR):
    # 1. Coerce to mono float32
    y = np.asarray(y, dtype=np.float32)
    if y.ndim > 1:
        # Handle both (channels, samples) and (samples, channels)
        # Average whichever axis has size 2 or less
        if y.shape[0] <= 2:
            y = y.mean(axis=0)
        else:
            y = y.mean(axis=1)

    # 2. Resample FIRST (some files are high-rate and become short)
    if sr != target_sr:
        y = librosa.resample(y, orig_sr=sr, target_sr=target_sr)
        sr = target_sr

    # 3. Trim
    try:
        y_trimmed, _ = librosa.effects.trim(y, top_db=30)
        if len(y_trimmed) > 0:
            y = y_trimmed
    except Exception:
        pass

    # 4. HARD length check - uses y.shape[0] explicitly, not len()
    min_samples = int(MIN_DURATION_SEC * sr)  # 11025 samples
    if y.shape[0] < min_samples:
        return None

    # 5. From here, we KNOW y is at least 11025 samples.
    #    hpss with n_fft=1024 will work (1024 << 11025).
    y_harm, _ = librosa.effects.hpss(y)

    chroma = librosa.feature.chroma_cqt(y=y_harm, sr=sr)
    chroma_mean = chroma.mean(axis=1)
    chroma_std = chroma.std(axis=1)

    mfcc = librosa.feature.mfcc(y=y_harm, sr=sr, n_mfcc=N_MFCC)
    mfcc_mean = mfcc.mean(axis=1)
    mfcc_std = mfcc.std(axis=1)

    contrast = librosa.feature.spectral_contrast(y=y_harm, sr=sr)
    contrast_mean = contrast.mean(axis=1)

    return np.concatenate([
        chroma_mean, chroma_std,
        mfcc_mean, mfcc_std,
        contrast_mean,
    ]).astype(np.float32)

In [ ]:
def build_dataset(pattern, verbose=True):
    X, Y = [], []
    dropped = 0
    for f in glob.glob(pattern):
        y, sr = librosa.load(f, sr=None)
        feat = extractChordFeatures(y, sr)
        if feat is None:
            if verbose:
                print(f"dropped (too short/silent): {f}")
            dropped+=1
            continue
        X.append(feat)
        Y.append(os.path.basename(os.path.dirname(f)))
    if verbose:
        print(f"Loaded {len(X)} files, dropped {dropped}")
    return np.array(X), np.array(Y)

In [ ]:
X_train, Y_train = build_dataset("./Training/*/*.wav")
X_test, Y_test = build_dataset("./Test/*/*.wav")
le = LabelEncoder()
y_train = le.fit_transform(Y_train)
y_test = le.transform(Y_test)

In [ ]:
clf = make_pipeline(
    StandardScaler(),
    SVC(kernel="rbf", C=10, gamma="scale", probability=True)
)
clf.fit(X_train, y_train)
print("Train acc:", clf.score(X_train, y_train))
print("Test  acc:", clf.score(X_test,  y_test))
print("CV 5-fold:", cross_val_score(clf, X_train, y_train, cv=5).mean())

In [ ]:
def testAudio(file, model, encoder, verbose=True):
    y, sr = librosa.load(file, sr=None) #function will handle the sr
    feat = extractChordFeatures(y, sr)
    if feat is None:
        return ("too short or silent", 0.0)
    feat = feat.reshape(1, -1)
    probs = model.predict_proba(feat)[0]
    idx = int(np.argmax(probs))
    chord = encoder.classes_[idx]
    confidence = float(probs[idx])

    #view top 3 for debugging

    if verbose:
        top3 = np.argsort(probs)[::-1][:3]
        ranked = ", ".join(
            f"{encoder.classes_[i]}={probs[i]:.2f}" for i in top3
        )
        print(f"{file}  →  {chord} ({confidence:.2f})   top3: [{ranked}]")

    return chord, confidence

MyPlays

In [ ]:
myPlays = sorted(glob.glob("./MyPlays/*.wav"))
print(f"Found {len(myPlays)} files\n")

results = []
for f in myPlays:
    chord, conf = testAudio(f, clf, le)
    results.append((f, chord, conf))